In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.6 Speculative decoding

A small **draft** model proposes several tokens cheaply; the big **target** model
verifies them in a single pass and accepts the longest correct prefix. When draft
and target agree often, you get the target's quality at a fraction of its steps,
with no change to the output distribution.

In [ ]:
from pocketlm import train_tiny, pick_config

corpus = open(CORPUS_PATH, encoding="utf-8").read()
target, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
draft, _ = train_tiny(corpus, pick_config(DEVICE, 1), seed=1)   # a different, weaker model
ctx = torch.tensor([tok.encode("The ")])
proposed = []
d = ctx.clone()
for _ in range(4):                       # draft proposes 4 tokens greedily
    dl, _ = draft(d[:, -draft.cfg.block_size:])
    nt = dl[:, -1].argmax(-1, keepdim=True)
    d = torch.cat([d, nt], dim=1)
    proposed.append(int(nt.item()))
accepted = 0
t = ctx.clone()
for tok_id in proposed:                  # target verifies greedily
    tl, _ = target(t[:, -target.cfg.block_size:])
    if int(tl[:, -1].argmax(-1).item()) == tok_id:
        accepted += 1
        t = torch.cat([t, torch.tensor([[tok_id]])], dim=1)
    else:
        break
print(f"draft proposed {len(proposed)}, target accepted {accepted}")

Every accepted token skipped a full target step. The target still has the final
say on each token, so correctness is preserved, the speedup is pure when agreement
is high.

In [ ]:
assert 0 <= accepted <= len(proposed)